In [3]:
import subprocess
import pandas as pd
import os, csv
import spacy, spacy_transformers
from string import punctuation
from wordfreq import zipf_frequency

C:\Users\dhima\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# def count_csv_elements_in_file(filepath):
#     total_elements=0
#     with open(filepath, 'r', encoding="latin-1") as file:
#         csv_reader=csv.reader(file)
#         for row in csv_reader:
#             total_elements+=len(row)
#     return total_elements
# language=[]
# total_words=[]
#
# for  path, subdir, files in os.walk('raw-word-lists'):
#     for name in files:
#         filepath = (os.path.join(path, name))
#         language+=[name.split('.')[0]]
#         total_words+=[count_csv_elements_in_file(filepath)]


In [5]:
# pd.DataFrame({'language':language,'total_words':total_words}).to_csv('clean-word-lists.csv',index=False)
df=pd.read_csv('clean-word-lists.csv')
df.drop(df.index[:28], inplace=True)
df.reset_index(inplace=True)
df.drop(columns=['index'], inplace=True)
df['language'].values

<StringArray>
[   'Catalan',   'Croatian',     'Danish',      'Dutch',    'English',
    'Finnish',     'French',     'German',      'Greek',    'Italian',
     'Polish', 'Portuguese',   'Romanian',    'Russian',  'Slovenian',
    'Spanish',    'Swedish',   'Ukranian']
Length: 18, dtype: str

In [6]:
spacy_models={
    'Catalan':"ca_core_news_trf",
    'Croatian':"hr_core_news_lg",
    'Danish':"da_core_news_trf",
    'Dutch':"nl_core_news_lg",
    'English':"en_core_web_trf",
    'Finnish':"fi_core_news_lg",
    'French':"fr_dep_news_trf",
    'German':"de_dep_news_trf",
    'Greek':"el_core_news_lg",
    'Italian':"it_core_news_lg",
     'Polish':"pl_core_news_lg",
    'Portuguese':"pt_co:re_news_lg",
    'Romanian':"ro_core_news_lg",
    'Russian':"ru_core_news_lg",
    'Slovenian':"sl_core_news_trf",
    'Spanish':"es_dep_news_trf",
    'Swedish':"sv_core_news_lg",
'Ukranian':"uk_core_news_trf"
}
venv_python = r"D:\PythonProject\.venv\Scripts\python.exe"
for model in spacy_models.values():
    result=subprocess.run([venv_python,'-m','spacy','download',f'{model}'],
                          capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)




Error processing line 7 of D:\PythonProject\.venv\Lib\site-packages\pywin32.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 206, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named 'pywin32_bootstrap'

Remainder of file ignored
Error processing line 7 of D:\PythonProject\.venv\Lib\site-packages\pywin32.pth:

  Traceback (most recent call last):
    File "<frozen site>", line 206, in addpackage
    File "<string>", line 1, in <module>
  ModuleNotFoundError: No module named 'pywin32_bootstrap'

Remainder of file ignored
Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "D:\PythonProject\.venv\Lib\site-packages\spacy\__init__.py", line 16, in <module>
    from .cli.info import info  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "D:\PythonProject\.venv\L

KeyboardInterrupt: 

In [ ]:
# import os
# import pandas as pd

In [ ]:
# making new dir for clean words
try:
    os.mkdir('data')
    for language in spacy_models.keys():
        try:
            os.mkdir(f'data/{language}')
            print(f"Directory {language} created")
        except FileExistsError:
            print(f'Directory {language} exits')
except FileExistsError:
    print(f'DATA dir  exits')



In [ ]:
def loan_and_clean_word_list(language:str) -> pd.DataFrame:
    with open(f'raw-word-lists/{language}/{language}.txt', 'r', encoding='latin-1') as f:
        word_list = f.read().split(',')
        word_df= pd.DataFrame({
        'word':word_list
    })
    word_df['word']= word_df['word'].str.strip(punctuation)
    return word_df

In [ ]:
# nlp=spacy.load(spacy_models[language], disable=['parser', 'tagger', 'ner'])
# getting the base word for the lang
def add_lemma(
        df: pd.DataFrame,
        nlp,
        batch_size:int=1000 ) -> pd.DataFrame:
    docs= nlp.pipe(df['word'].tolist(), batch_size=batch_size )
    lemmas = [doc[0].lemma_ for doc in docs]
    df['lemma'] = pd.DataFrame(lemmas, index=df.index)
    return df
# getting how often word is being used i.e lemma
def word_frequency(
        df: pd.DataFrame,
        language:str
)->pd.DataFrame:
    language_group= spacy_models['language'].strip('_')[0]
    df['zipf_freq_lemma']=[zipf_frequency(word, lang=language) for word in df['lemma']]
    return df
def clean_up_and_export(
        df: pd.DataFrame,
        language:str
)-> None:
    df=(
        df.loc[df.groupby('lemma', sort=False)['zipf_freq_lemma'].idxmax()].reset_index(drop=True)
    )
    df=df[df['zipf_freq_lemma']>0]
    df.loc[:, 'word_difficulty']=pd.cut(
        df['zipf_freq_lemma'], bins=[-float('inf'),2.0,4.0 ,float('inf')],
        labels=['advanced', 'intermediate', 'beginnner'],include_lowest=True, right=True
    )
    df.drop(columns=['zipf_freq_lemma','word'], inplace=True)
    df.rename(columns={'lemma':'word'})
    df.to_json(f'data/{language}/word-list-cleaned',orient='index')


In [ ]:
def create_clean_word_list(language:str) -> None:
    nlp=spacy.load(spacy_models[language], disable=['parser', 'textcar', 'ner'])
    print('load in data')
    lang_df=loan_and_clean_word_list(language)
    print('lemmatize word')
    lang_df= add_lemma(lang_df, nlp)
    print('adding zipf freq')
    lang_df=word_frequency(lang_df, language)
    print('do the final clean ups and export')
    clean_up_and_export(lang_df, language)
    return None


In [ ]:
create_clean_word_list('Spanish')